In [1]:
# --- Célula de Retreino Completo: Otimização e Salvamento dos Modelos Finais ---

# Instala as bibliotecas necessárias
!pip install -q pandas numpy catboost joblib optuna

import pandas as pd
import numpy as np
import catboost as cat
import joblib
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report
import optuna
import warnings

# Configurações do ambiente
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING) # Deixa o Optuna menos verboso
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# --- ETAPA 1: CARREGAMENTO E PREPARAÇÃO DOS DADOS ---
try:
    df = pd.read_csv('aluminium_full_featured_retrained.csv', parse_dates=['date'])
    print(f"Dataset 'aluminium_full_featured_retrained.csv' carregado com sucesso.")
    print(f"Período dos dados para treino: de {df['date'].min().date()} a {df['date'].max().date()}")

    # Transforma os dados para o formato 'long'
    base_features = [col for col in df.columns if col not in ['date'] and not any(s in col for s in ['bin_', 'ret_', 'future_'])]
    records = [
        df[base_features + ['date', f'bin_{h}d']]
        .rename(columns={f'bin_{h}d': 'target'})
        .assign(horizon=h)
        .dropna(subset=['target'])
        for h in range(1, 91) if f'bin_{h}d' in df.columns
    ]
    df_long = pd.concat(records, ignore_index=True)
    print("DataFrame transformado para o formato 'long'.")

except FileNotFoundError:
    print("ERRO: Ficheiro 'aluminium_full_featured_retrained.csv' não encontrado.")
    print("Por favor, certifique-se de que o upload do ficheiro foi concluído.")
    df_long = None

# --- ETAPA 2: DEFINIÇÃO DA FUNÇÃO DE OTIMIZAÇÃO ---
if df_long is not None:
    def objective_cat(trial, X_train, y_train, X_val, y_val):
        categorical_features_indices = [
            X_train.columns.get_loc(c) for c in 
            ['month', 'quarter', 'weekday', 'end_of_month'] 
            if c in X_train.columns
        ]
        params = {
            'iterations': trial.suggest_int('iterations', 200, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
            'depth': trial.suggest_int('depth', 3, 12),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
            'random_state': RANDOM_STATE, 'verbose': 0,
            'cat_features': categorical_features_indices, 'auto_class_weights': 'Balanced'
        }
        model = cat.CatBoostClassifier(**params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=15, verbose=False)
        return f1_score(y_val, model.predict(X_val), average='macro', zero_division=0)

    print("Função de otimização definida.")

# --- ETAPA 3: LOOP DE OTIMIZAÇÃO, TREINO E SALVAMENTO ---
if df_long is not None:
    horizons_to_model = [1, 5, 30, 90]
    
    for hz in horizons_to_model:
        print(f"\n=============================================================")
        print(f" INICIANDO RETREINO COMPLETO PARA HORIZONTE: {hz} DIAS")
        print(f"=============================================================")

        # 1. Prepara os dados para o horizonte atual
        df_horizon = df_long[df_long['horizon'] == hz].copy()
        label_encoder = LabelEncoder()
        df_horizon['target_encoded'] = label_encoder.fit_transform(df_horizon['target'])
        feature_cols = [c for c in df_horizon.columns if c not in ['target', 'date', 'target_encoded']]
        
        # Divisão temporal: 80% para treino/validação, 20% para teste final
        df_horizon_sorted = df_horizon.sort_values('date').reset_index(drop=True)
        split_index = int(0.8 * len(df_horizon_sorted))
        train_val_df = df_horizon_sorted.iloc[:split_index]
        holdout_df = df_horizon_sorted.iloc[split_index:]
        
        X_train_val, y_train_val = train_val_df[feature_cols], train_val_df['target_encoded']
        X_holdout, y_holdout = holdout_df[feature_cols], holdout_df['target_encoded']
        
        print(f"  - Dados preparados. Treino/Validação: {len(train_val_df)}, Teste Final: {len(holdout_df)}")

        # 2. Executa a Otimização com Optuna, salvando o progresso
        print(f"  - A iniciar a otimização de hiperparâmetros (50 tentativas)...")
        study_db_path = f"sqlite:///catboost_study_retrain_h{hz}.db"
        study = optuna.create_study(direction='maximize', study_name=f"catboost_final_h{hz}", storage=study_db_path, load_if_exists=True)
        study.optimize(lambda trial: objective_cat(trial, X_train_val, y_train_val, X_holdout, y_holdout), 
                       n_trials=50, timeout=1200)
        
        best_params = study.best_params
        print(f"  - Otimização concluída. Melhor F1-Score (validação): {study.best_value:.4f}")

        # 3. Salva os melhores hiperparâmetros em JSON
        params_filename = f'best_catboost_params_h{hz}_retrained.json'
        with open(params_filename, 'w') as f:
            json.dump(best_params, f, indent=4)
        print(f"  - Melhores hiperparâmetros salvos em '{params_filename}'")
        
        # 4. Treina o Modelo Final com TODOS os dados
        print("  - A treinar o modelo final com todos os dados...")
        X_full, y_full = df_horizon_sorted[feature_cols], df_horizon_sorted['target_encoded']
        categorical_features_indices = [X_full.columns.get_loc(c) for c in ['month', 'quarter', 'weekday', 'end_of_month'] if c in X_full.columns]
        
        final_params = {**best_params, 'random_state': RANDOM_STATE, 'verbose': 0, 'cat_features': categorical_features_indices, 'auto_class_weights': 'Balanced'}
        final_model = cat.CatBoostClassifier(**final_params)
        final_model.fit(X_full, y_full)

        # 5. Salva os Artefatos para Análise e Produção
        print("  - A salvar o modelo, encoder e dataframes...") 
        joblib.dump(final_model, f'catboost_model_horizon_{hz}d_retrained.joblib')
        joblib.dump(label_encoder, f'label_encoder_horizon_{hz}d_retrained.joblib')
        joblib.dump(holdout_df, f'test_df_horizon_{hz}d_retrained.joblib')
        joblib.dump(train_val_df, f'train_df_horizon_{hz}d_retrained.joblib')

print("\n\n Processo completo de retreino, otimização e salvamento concluído para todos os horizontes!")

Dataset 'aluminium_full_featured_retrained.csv' carregado com sucesso.
Período dos dados para treino: de 2020-09-29 a 2025-08-29
DataFrame transformado para o formato 'long'.
Função de otimização definida.

 INICIANDO RETREINO COMPLETO PARA HORIZONTE: 1 DIAS
  - Dados preparados. Treino/Validação: 1028, Teste Final: 257
  - A iniciar a otimização de hiperparâmetros (50 tentativas)...
  - Otimização concluída. Melhor F1-Score (validação): 0.2965
  - Melhores hiperparâmetros salvos em 'best_catboost_params_h1_retrained.json'
  - A treinar o modelo final com todos os dados...
  - A salvar o modelo, encoder e dataframes...

 INICIANDO RETREINO COMPLETO PARA HORIZONTE: 5 DIAS
  - Dados preparados. Treino/Validação: 1028, Teste Final: 257
  - A iniciar a otimização de hiperparâmetros (50 tentativas)...
  - Otimização concluída. Melhor F1-Score (validação): 0.2157
  - Melhores hiperparâmetros salvos em 'best_catboost_params_h5_retrained.json'
  - A treinar o modelo final com todos os dados...